# StoryForge on Colab
Enable **Runtime → Change runtime type → T4 GPU**, then upload the supplied ZIP. Free GPU availability and session duration are not guaranteed.

In [ ]:
from google.colab import files
uploaded = files.upload()
import zipfile, pathlib
name = next(n for n in uploaded if n.endswith(".zip"))
with zipfile.ZipFile(name) as z:
    z.extractall("/content")
%cd /content/storyforge
%pip install -r requirements.txt

In [ ]:
import torch
print(torch.__version__, "CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime before full training"
!python -m pytest -q

Mount Drive to retain checkpoints through runtime resets. Tokenized data remains in this runtime; preserve it too if you need to resume without rebuilding it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!python prepare_data.py --out /content/drive/MyDrive/storyforge/data --train-stories 100000 --val-stories 5000

In [ ]:
!python train.py --data /content/drive/MyDrive/storyforge/data --out /content/drive/MyDrive/storyforge/run --steps 5000 --batch-size 16 --accumulation 4

If interrupted, rerun training with the same arguments plus `--resume /content/drive/MyDrive/storyforge/run/last.pt`. Skip preparation if its metadata already exists.

In [ ]:
!python generate.py --checkpoint /content/drive/MyDrive/storyforge/run/best.pt --device cuda --prompt "A little fox found a key"
!python benchmark.py --checkpoint /content/drive/MyDrive/storyforge/run/best.pt --device cuda

In [ ]:
import shutil
shutil.make_archive("/content/storyforge_trained_run", "zip", "/content/drive/MyDrive/storyforge/run")
files.download("/content/storyforge_trained_run.zip")